In [1]:
import pandas as pd

df = pd.read_csv("dataset.csv")
df.replace("NULL", pd.NA, inplace=True)

# Rows with no missing values
complete_dataset = df[df.notna().all(axis=1)]

# Rows with one or more missing values
missing_dataset = df[df.isna().any(axis=1)]

complete_dataset.to_csv("complete_dataset.csv", index=False)
missing_dataset.to_csv("missing_dataset.csv", index=False)
print(complete_dataset)
print(missing_dataset)
#this part is to split the dataset into 2 part

    Col_1  Col_2  Col_3  Col_4  Col_5  Col_6  Col_7  Col_8  Col_9  Col_10
0    77.2   23.4   51.6   88.1   12.9   45.3   67.8   91.2   34.5    56.7
1    14.8   89.1   33.4   56.2   78.9   11.5   44.2   83.1   22.6    95.4
2    62.1   41.7   85.0   29.3   66.4   73.1   18.9   50.5   87.3    10.2
3    39.5   72.3   11.8   94.6   53.2   81.0   27.4   68.9   49.1    36.8
4    88.4   19.2   67.5   43.1   90.7   25.6   58.3   12.4   76.9    82.0
5    21.9   55.6   92.3   77.8   34.1   69.4   85.2   39.7   14.0    61.3
6    50.3   83.8   44.1   12.5   57.9   98.2   31.6   74.3   63.8    29.5
7    96.7   36.4   78.2   65.9   22.1   47.5   93.0   58.1   41.2    73.6
8    42.6   68.1   25.9   81.4   89.3   13.7   70.4   26.5   95.8    50.1
9    73.5   91.0   59.4   37.2   46.8   82.3   52.1   69.0   11.4    88.2
10   11.2   24.7   83.9   50.6   71.4   36.9   64.5   87.8   53.1    19.6
11   65.4   79.3   16.2   93.8   38.5   61.2   29.1   42.7   80.6    75.3
12   84.1   52.8   70.6   21.9   95.2 

In [2]:
import pandas as pd
import numpy as np
import random

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, RegressorMixin

from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

# Framework for your custom DNN algorithm
import torch
import torch.nn as nn
import torch.optim as optim

# Optional libraries
try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False


# ==============================================================================
# SELF-MADE ALGORITHM: GENETIC ALGORITHM OPTIMIZED DEEP NEURAL NETWORK
# ==============================================================================
class DynamicPyTorchDNN(nn.Module):
    """Dynamically builds a Neural Network structure based on GA parameters."""
    def __init__(self, input_dim, num_layers, hidden_units, activation_str):
        super().__init__()
        layers = []
        in_dim = input_dim

        if activation_str == 'relu':
            act_fn = nn.ReLU()
        elif activation_str == 'tanh':
            act_fn = nn.Tanh()
        else:
            act_fn = nn.Sigmoid()

        for _ in range(num_layers):
            layers.append(nn.Linear(in_dim, hidden_units))
            layers.append(act_fn)
            in_dim = hidden_units

        layers.append(nn.Linear(in_dim, 1)) # Continuous target for regression
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(-1)


class GeneticDNNRegressor(BaseEstimator, RegressorMixin):
    """Custom Scikit-Learn Regressor driven by a Genetic Algorithm."""
    def __init__(self, generations=3, population_size=4, epochs=15, random_state=42):
        self.generations = generations
        self.population_size = population_size
        self.epochs = epochs
        self.random_state = random_state
        self.best_model_ = None
        self.best_hyperparameters_ = None

    def _random_chromosome(self):
        # Genotype Space: [Layers, Neurons, Learning Rate, Activation]
        return [
            random.choice([1, 2, 3]),
            random.choice([16, 32, 64, 128]),
            random.choice([0.001, 0.01, 0.05]),
            random.choice(['relu', 'tanh', 'sigmoid'])
        ]

    def _evaluate_fitness(self, chromosome, X_tr, y_tr, X_val, y_val):
        num_layers, hidden_units, lr, activation = chromosome
        input_dim = X_tr.shape[1]

        model = DynamicPyTorchDNN(input_dim, num_layers, hidden_units, activation)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr)

        X_tr_t = torch.FloatTensor(X_tr)
        y_tr_t = torch.FloatTensor(y_tr)
        X_val_t = torch.FloatTensor(X_val)
        y_val_t = torch.FloatTensor(y_val)

        # Intermediate mini-training step
        model.train()
        for epoch in range(self.epochs):
            optimizer.zero_grad()
            loss = criterion(model(X_tr_t), y_tr_t)
            loss.backward()
            optimizer.step()

        # Fitness based on validation error
        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_val_t), y_val_t).item()
        return val_loss

    def fit(self, X, y):
        # Convert DataFrames/Series to clean numpy arrays
        X_arr = X.values if hasattr(X, 'values') else np.array(X)
        y_arr = y.values if hasattr(y, 'values') else np.array(y)

        # Internal holdout sets to evaluate hyperparameter fitness safely
        split = int(len(X_arr) * 0.8)
        X_tr, X_val = X_arr[:split], X_arr[split:]
        y_tr, y_val = y_arr[:split], y_arr[split:]

        random.seed(self.random_state)
        torch.manual_seed(self.random_state)

        # Initialize Base Population
        population = [self._random_chromosome() for _ in range(self.population_size)]
        best_loss = float('inf')

        for gen in range(self.generations):
            scores = [self._evaluate_fitness(chrom, X_tr, y_tr, X_val, y_val) for chrom in population]

            # Record absolute best configuration
            min_idx = np.argmin(scores)
            if scores[min_idx] < best_loss:
                best_loss = scores[min_idx]
                self.best_hyperparameters_ = population[min_idx]

            # Evolve population (Selection via sorting)
            sorted_indices = np.argsort(scores)
            population = [population[i] for i in sorted_indices]

            # Generate next generation (Keep top 2 Elites)
            next_gen = [population[0], population[1]]

            while len(next_gen) < self.population_size:
                p1, p2 = population[random.randint(0, 1)], population[random.randint(0, min(3, len(population)-1))]
                # Single-point Crossover
                child = [p1[0], p2[1], p1[2] if random.random() > 0.5 else p2[2], p1[3] if random.random() > 0.5 else p2[3]]
                # Mutation
                if random.random() < 0.2:
                    child[random.randint(0, 3)] = self._random_chromosome()[random.randint(0, 3)]
                next_gen.append(child)

            population = next_gen

        # Final Step: Retrain top evolving configuration on full dataset slice
        num_layers, hidden_units, lr, activation = self.best_hyperparameters_
        self.best_model_ = DynamicPyTorchDNN(X_arr.shape[1], num_layers, hidden_units, activation)

        X_t, y_t = torch.FloatTensor(X_arr), torch.FloatTensor(y_arr)
        optimizer = optim.Adam(self.best_model_.parameters(), lr=lr)
        criterion = nn.MSELoss()

        self.best_model_.train()
        for epoch in range(self.epochs * 2): # Train slightly longer on total subset
            optimizer.zero_grad()
            loss = criterion(self.best_model_(X_t), y_t)
            loss.backward()
            optimizer.step()
        return self

    def predict(self, X):
        X_arr = X.values if hasattr(X, 'values') else np.array(X)
        self.best_model_.eval()
        with torch.no_grad():
            preds = self.best_model_(torch.FloatTensor(X_arr))
        return preds.numpy()


# ==============================================================================
# MAIN PIPELINE RUNNER
# ==============================================================================
DATASET_PATH = "complete_dataset.csv"
TEST_SIZE = 0.20
RANDOM_STATE = 42

df = pd.read_csv(DATASET_PATH)
df = df.select_dtypes(include=[np.number])

print("=" * 100)
print(f"Dataset Shape: {df.shape}")
print("Columns:", list(df.columns))
print("=" * 100)

results = []

for target_col in df.columns:

    print("\n" + "#" * 100)
    print(f"TARGET COLUMN: {target_col}")
    print("#" * 100)

    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE
    )

    models = {
        "AdaBoost": AdaBoostRegressor(random_state=RANDOM_STATE),
        "KNN": KNeighborsRegressor(n_neighbors=min(5, len(X_train))),
        "RandomForest": RandomForestRegressor(
            n_estimators=200,
            random_state=RANDOM_STATE
        ),
        "DecisionTree": DecisionTreeRegressor(
            random_state=RANDOM_STATE
        ),
        # INTEGRATED CUSTOM GENETIC ALGORITHM DNN
        "GeneticDNN": GeneticDNNRegressor(
            generations=3,
            population_size=4,
            epochs=15,
            random_state=RANDOM_STATE
        )
    }

    if XGB_AVAILABLE:
        models["XGBoost"] = XGBRegressor(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            random_state=RANDOM_STATE,
            objective="reg:squarederror"
        )

    if LGBM_AVAILABLE:
        models["LightGBM"] = LGBMRegressor(
            n_estimators=200,
            learning_rate=0.05,
            random_state=RANDOM_STATE,
            verbose=-1
        )

    column_results = []

    for model_name, model in models.items():

        pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("model", model)
        ])

        pipeline.fit(X_train, y_train)
        predictions = pipeline.predict(X_test)

        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        r2 = r2_score(y_test, predictions)

        column_results.append({
            "Algorithm": model_name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

        print(f"\n{model_name}")
        print(f"MAE  = {mae:.6f}")
        print(f"RMSE = {rmse:.6f}")
        print(f"R²   = {r2:.6f}")

        comparison = pd.DataFrame({
            "Actual": y_test.values,
            "Predicted": predictions,
            "Absolute_Error": np.abs(y_test.values - predictions)
        })

        print("\nActual vs Predicted (first 10 rows):")
        print(comparison.head(10))

        results.append({
            "Target_Column": target_col,
            "Algorithm": model_name,
            "MAE": mae,
            "RMSE": rmse,
            "R2": r2
        })

    ranking = pd.DataFrame(column_results)
    ranking = ranking.sort_values("RMSE")

    print("\n" + "-" * 80)
    print(f"BEST MODELS FOR {target_col}")
    print("-" * 80)
    print(ranking)

    print(f"\nBest Algorithm for {target_col}: "
          f"{ranking.iloc[0]['Algorithm']} "
          f"(RMSE={ranking.iloc[0]['RMSE']:.6f})")

final_results = pd.DataFrame(results)

print("\n\n" + "=" * 100)
print("OVERALL RESULTS")
print("=" * 100)

print(
    final_results
    .sort_values(["Target_Column", "RMSE"])
)

final_results.to_csv("all_column_algorithm_comparison.csv", index=False)
print("\nSaved: all_column_algorithm_comparison.csv")

Dataset Shape: (20, 10)
Columns: ['Col_1', 'Col_2', 'Col_3', 'Col_4', 'Col_5', 'Col_6', 'Col_7', 'Col_8', 'Col_9', 'Col_10']

####################################################################################################
TARGET COLUMN: Col_1
####################################################################################################

AdaBoost
MAE  = 36.412500
RMSE = 38.720707
R²   = -1.147715

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    77.2      33.70           43.50
1    81.6      42.60           39.00
2    57.0      42.60           14.40
3    14.8      63.55           48.75

KNN
MAE  = 33.920000
RMSE = 36.593456
R²   = -0.918214

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    77.2      40.60           36.60
1    81.6      50.38           31.22
2    57.0      42.30           14.70
3    14.8      67.96           53.16

RandomForest
MAE  = 26.413000
RMSE = 30.311494
R²   = -0.316148

Actual vs Predicted (f

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



AdaBoost
MAE  = 25.550000
RMSE = 29.286681
R²   = 0.097834

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    23.4     41.925          18.525
1    31.5     21.950           9.550
2    88.6     40.375          48.225
3    89.1     63.200          25.900

KNN
MAE  = 29.170000
RMSE = 31.216540
R²   = -0.024981

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    23.4      58.88           35.48
1    31.5      42.70           11.20
2    88.6      47.96           40.64
3    89.1      59.74           29.36

RandomForest
MAE  = 27.526750
RMSE = 30.725032
R²   = 0.007042

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    23.4    45.6515         22.2515
1    31.5    40.2650          8.7650
2    88.6    42.8700         45.7300
3    89.1    55.7395         33.3605

DecisionTree
MAE  = 37.150000
RMSE = 45.950680
R²   = -1.220906

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    23.4  

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



RandomForest
MAE  = 17.881625
RMSE = 23.625393
R²   = -0.252568

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    51.6    52.9820          1.3820
1    64.3    51.0465         13.2535
2    91.4    48.1765         43.2235
3    33.4    47.0675         13.6675

DecisionTree
MAE  = 25.200000
RMSE = 28.125700
R²   = -0.775211

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    51.6       16.2            35.4
1    64.3       25.9            38.4
2    91.4       83.9             7.5
3    33.4       13.9            19.5

GeneticDNN
MAE  = 37.721565
RMSE = 43.250648
R²   = -3.197865

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    51.6  23.106331       28.493669
1    64.3  23.597654       40.702346
2    91.4  21.595802       69.804198
3    33.4  21.513952       11.886048

XGBoost
MAE  = 23.652549
RMSE = 27.639014
R²   = -0.714306

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0  

...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



RandomForest
MAE  = 11.539125
RMSE = 12.575725
R²   = -1.317985

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    34.5    50.1225         15.6225
1    45.7    52.7750          7.0750
2    37.1    43.2030          6.1030
3    22.6    39.9560         17.3560

DecisionTree
MAE  = 22.075000
RMSE = 24.482800
R²   = -7.785504

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    34.5       11.4            23.1
1    45.7       11.4            34.3
2    37.1       11.4            25.7
3    22.6       17.4             5.2

GeneticDNN
MAE  = 28.460863
RMSE = 29.635243
R²   = -11.872459

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    34.5  63.435863       28.935863
1    45.7  63.435863       17.735863
2    37.1  63.435863       26.335863
3    22.6  63.435863       40.835863

XGBoost
MAE  = 12.050978
RMSE = 17.170516
R²   = -3.321268

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



RandomForest
MAE  = 29.799125
RMSE = 32.559971
R²   = -3.931741

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    56.7    77.2270         20.5270
1    83.4    65.5900         17.8100
2    68.5    38.8620         29.6380
3    95.4    44.1785         51.2215

DecisionTree
MAE  = 31.150000
RMSE = 35.261665
R²   = -4.784128

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    56.7       91.7            35.0
1    83.4       75.3             8.1
2    68.5       41.2            27.3
3    95.4       41.2            54.2

GeneticDNN
MAE  = 52.542743
RMSE = 55.317232
R²   = -13.234857

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0    56.7  25.381493       31.318507
1    83.4  23.581501       59.818499
2    68.5  26.250620       42.249380
3    95.4  18.615414       76.784586

XGBoost
MAE  = 33.602465
RMSE = 37.484665
R²   = -5.536413

Actual vs Predicted (first 10 rows):
   Actual  Predicted  Absolute_Error
0 

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [ ]:

import pandas as pd
import numpy as np

from sklearn.impute import SimpleImputer
from sklearn.ensemble import AdaBoostRegressor, RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

MISSING_DATASET = "missing_dataset.csv"
COMPARISON_FILE = "all_column_algorithm_comparison.csv"

df = pd.read_csv(MISSING_DATASET)
comparison = pd.read_csv(COMPARISON_FILE)

filled_df = df.copy()

# Select best algorithm (lowest RMSE) for each target column
best_models = (
    comparison.sort_values("RMSE")
    .groupby("Target_Column", as_index=False)
    .first()
)

print("\nBEST ALGORITHM FOR EACH COLUMN")
print("=" * 80)
print(best_models[["Target_Column", "Algorithm", "RMSE"]])

def build_model(name):
    if name == "AdaBoost":
        return AdaBoostRegressor(random_state=42)

    if name == "KNN":
        return KNeighborsRegressor(n_neighbors=5)

    if name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=200,
            random_state=42
        )

    if name == "DecisionTree":
        return DecisionTreeRegressor(random_state=42)

    if name == "XGBoost" and XGB_AVAILABLE:
        return XGBRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            random_state=42,
            objective="reg:squarederror"
        )

    if name == "LightGBM" and LGBM_AVAILABLE:
        return LGBMRegressor(
            n_estimators=200,
            learning_rate=0.05,
            random_state=42,
            verbose=-1
        )

    raise ValueError(f"Model not available: {name}")


# Multiple passes help when rows contain several missing values
for iteration in range(3):

    print(f"\n========== IMPUTATION PASS {iteration + 1} ==========")

    for _, row in best_models.iterrows():

        target = row["Target_Column"]
        algo = row["Algorithm"]

        if target not in filled_df.columns:
            continue

        missing_mask = filled_df[target].isna()

        if missing_mask.sum() == 0:
            continue

        print(
            f"Column: {target:<20} "
            f"Algorithm: {algo:<15} "
            f"Missing Values: {missing_mask.sum()}"
        )

        train_mask = filled_df[target].notna()

        feature_cols = [c for c in filled_df.columns if c != target]

        X_train = filled_df.loc[train_mask, feature_cols]
        y_train = filled_df.loc[train_mask, target]

        X_missing = filled_df.loc[missing_mask, feature_cols]

        # IMPORTANT:
        # Features may contain missing values.
        # We impute FEATURES only.
        imputer = SimpleImputer(strategy="mean")

        X_train_imp = imputer.fit_transform(X_train)
        X_missing_imp = imputer.transform(X_missing)

        model = build_model(algo)

        model.fit(X_train_imp, y_train)

        predicted_values = model.predict(X_missing_imp)

        filled_df.loc[missing_mask, target] = predicted_values

print("\n\nFINAL DATASET AFTER IMPUTATION")
print("=" * 100)
print(filled_df)

filled_df.to_csv("missing_dataset_filled.csv", index=False)

print("\nSaved file: missing_dataset_filled.csv")



BEST ALGORITHM FOR EACH COLUMN
  Target_Column     Algorithm       RMSE
0         Col_1      LightGBM  26.469370
1        Col_10      LightGBM  28.568061
2         Col_2       XGBoost  28.032293
3         Col_3      LightGBM  22.410656
4         Col_4  DecisionTree  21.967817
5         Col_5      AdaBoost  24.288343
6         Col_6       XGBoost  26.981103
7         Col_7      LightGBM  23.345466
8         Col_8      AdaBoost  20.918906
9         Col_9  RandomForest  12.575725

========== IMPUTATION PASS 1 ==========
Column: Col_1                Algorithm: LightGBM        Missing Values: 14
Column: Col_10               Algorithm: LightGBM        Missing Values: 13
Column: Col_2                Algorithm: XGBoost         Missing Values: 14


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Column: Col_3                Algorithm: LightGBM        Missing Values: 14
Column: Col_4                Algorithm: DecisionTree    Missing Values: 13
Column: Col_5                Algorithm: AdaBoost        Missing Values: 13
Column: Col_6                Algorithm: XGBoost         Missing Values: 13
Column: Col_7                Algorithm: LightGBM        Missing Values: 12
Column: Col_8                Algorithm: AdaBoost        Missing Values: 13
Column: Col_9                Algorithm: RandomForest    Missing Values: 12


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(



========== IMPUTATION PASS 2 ==========

========== IMPUTATION PASS 3 ==========


FINAL DATASET AFTER IMPUTATION
       Col_1      Col_2     Col_3  Col_4      Col_5      Col_6      Col_7  \
0   25.40000  44.412819  54.80000   71.2  48.510000  39.100000  52.333333   
1   56.64375  44.900000  54.90625   19.3  82.400000  49.497543  53.600000   
2   61.30000  88.000000  54.90625   26.8  41.700000  90.300000  22.500000   
3   56.64375  65.789589  31.60000   64.8  15.200000  52.155762  79.100000   
4   79.20000  13.500000  88.10000   15.6  50.400000  62.700000  52.333333   
5   43.00000  58.800419  27.40000   95.1  68.300000  39.866959  14.900000   
6   56.64375  76.400000  54.90625   38.0  18.183333  51.600000  88.200000   
7   87.50000  29.100000  62.90000   59.2  50.400000  73.800000  52.333333   
8   56.64375  22.157545  94.20000   11.7  49.675000  55.788181  43.000000   
9   52.10000  81.600000  54.90625   87.3  36.900000  19.500000  52.333333   
10  56.64375  50.200000  18.50000   64